# Merge 2018-2019 Floors 3, 4, 5, and 7

This notebook is a focused version of the merge workflow. It only merges the raw CSV files for floors 3, 4, 5, and 7 across 2018 and 2019.

Scope:

- Years: `2018`, `2019`
- Floors: `3`, `4`, `5`, `7`
- Input folder: `Raw Data/`
- Output folder: `outputs/`
- Main output: `combined_wide_2018_2019_floors_3_4_5_7.csv.gz`

The merge is vertical concatenation. It keeps `year`, `floor`, and `source_file` columns so every row can be traced back to its original CSV file.


## 1. Import libraries and locate project folders


In [1]:
from pathlib import Path
from typing import Optional
import re

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_NAME = "Artificial Intelligence for Smart-Building Log Analysis"
YEARS = [2018, 2019]
FLOORS = [3, 4, 5, 7]


def find_project_dir(start: Optional[Path] = None) -> Path:
    """Find the project folder from the current notebook location or parent folders."""
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if path.name == PROJECT_NAME and (path / "Raw Data").exists():
            return path
        candidate = path / PROJECT_NAME
        if (candidate / "Raw Data").exists():
            return candidate
    raise FileNotFoundError(f"Could not find '{PROJECT_NAME}/Raw Data'.")


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "Raw Data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_DIR, DATA_DIR, OUTPUT_DIR


(PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis'),
 PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/Raw Data'),
 PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs'))

## 2. Select the required 8 source CSV files


In [2]:
source_files = [DATA_DIR / f"{year}Floor{floor}.csv" for year in YEARS for floor in FLOORS]
missing_files = [path for path in source_files if not path.exists()]

if missing_files:
    raise FileNotFoundError("Missing required CSV files:\n" + "\n".join(str(path) for path in missing_files))

source_file_table = pd.DataFrame({
    "year": [int(re.search(r"(20\d{2})", path.name).group(1)) for path in source_files],
    "floor": [int(re.search(r"Floor(\d+)", path.name).group(1)) for path in source_files],
    "file_name": [path.name for path in source_files],
    "file_path": [str(path) for path in source_files],
})

source_file_table


,year,floor,file_name,file_path
0,2018,3,2018Floor3.csv,/Users/yonglijiang/Documents/Adelaide Universi...
1,2018,4,2018Floor4.csv,/Users/yonglijiang/Documents/Adelaide Universi...
2,2018,5,2018Floor5.csv,/Users/yonglijiang/Documents/Adelaide Universi...
3,2018,7,2018Floor7.csv,/Users/yonglijiang/Documents/Adelaide Universi...
4,2019,3,2019Floor3.csv,/Users/yonglijiang/Documents/Adelaide Universi...
5,2019,4,2019Floor4.csv,/Users/yonglijiang/Documents/Adelaide Universi...
6,2019,5,2019Floor5.csv,/Users/yonglijiang/Documents/Adelaide Universi...
7,2019,7,2019Floor7.csv,/Users/yonglijiang/Documents/Adelaide Universi...


## 3. Check column counts and field consistency

Floors 3, 4, 5, and 7 should use the same 30 fields. This check prevents accidental merging by position when field names do not match.


In [3]:
headers = {}
for path in source_files:
    headers[path.name] = list(pd.read_csv(path, nrows=0).columns)

reference_file = source_files[0].name
reference_columns = headers[reference_file]

column_check = pd.DataFrame({
    "file_name": list(headers.keys()),
    "column_count": [len(cols) for cols in headers.values()],
    "same_columns_and_order_as_reference": [cols == reference_columns for cols in headers.values()],
})

COLUMN_CHECK_PATH = OUTPUT_DIR / "column_check_2018_2019_floors_3_4_5_7.csv"
column_check.to_csv(COLUMN_CHECK_PATH, index=False)

print("Reference file:", reference_file)
print("Reference column count:", len(reference_columns))
print("Column check saved to:", COLUMN_CHECK_PATH)

if not column_check["same_columns_and_order_as_reference"].all():
    mismatched = column_check.loc[~column_check["same_columns_and_order_as_reference"], "file_name"].tolist()
    raise ValueError(f"Column mismatch found in: {mismatched}")

column_check


Reference file: 2018Floor3.csv
Reference column count: 30
Column check saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/column_check_2018_2019_floors_3_4_5_7.csv


,file_name,column_count,same_columns_and_order_as_reference
0,2018Floor3.csv,30,True
1,2018Floor4.csv,30,True
2,2018Floor5.csv,30,True
3,2018Floor7.csv,30,True
4,2019Floor3.csv,30,True
5,2019Floor4.csv,30,True
6,2019Floor5.csv,30,True
7,2019Floor7.csv,30,True


## 4. Show the 30 columns used in the merge


In [4]:
pd.DataFrame({"column_index": range(1, len(reference_columns) + 1), "column_name": reference_columns})


,column_index,column_name
0,1,Date
1,2,z1_AC1(kW)
2,3,z1_AC2(kW)
3,4,z1_AC3(kW)
4,5,z1_AC4(kW)
5,6,z1_Light(kW)
6,7,z1_Plug(kW)
7,8,z1_S1(degC)
8,9,z1_S1(RH%)
9,10,z1_S1(lux)


## 5. Count rows before merging


In [5]:
row_counts = []
for path in source_files:
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        rows = sum(1 for _ in handle) - 1
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))
    row_counts.append({"year": year, "floor": floor, "file_name": path.name, "row_count": rows})

row_count_table = pd.DataFrame(row_counts)
ROW_COUNT_PATH = OUTPUT_DIR / "row_counts_2018_2019_floors_3_4_5_7.csv"
row_count_table.to_csv(ROW_COUNT_PATH, index=False)

print("Expected merged rows:", f"{row_count_table['row_count'].sum():,}")
print("Row count summary saved to:", ROW_COUNT_PATH)

row_count_table


Expected merged rows: 3,162,240
Row count summary saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/row_counts_2018_2019_floors_3_4_5_7.csv


,year,floor,file_name,row_count
0,2018,3,2018Floor3.csv,264960
1,2018,4,2018Floor4.csv,264960
2,2018,5,2018Floor5.csv,264960
3,2018,7,2018Floor7.csv,264960
4,2019,3,2019Floor3.csv,525600
5,2019,4,2019Floor4.csv,525600
6,2019,5,2019Floor5.csv,525600
7,2019,7,2019Floor7.csv,525600


## 6. Merge the selected CSV files

This section reads each selected raw file, standardizes the `Date` column to datetime, adds traceability columns, and vertically concatenates the tables.


In [6]:
merged_parts = []

for path in source_files:
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))

    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df.insert(0, "source_file", path.name)
    df.insert(0, "floor", floor)
    df.insert(0, "year", year)
    merged_parts.append(df)

combined = pd.concat(merged_parts, ignore_index=True)
combined = combined.sort_values(["year", "floor", "Date"]).reset_index(drop=True)

combined.shape


(3162240, 33)

## 7. Validate the merged table


In [7]:
validation_summary = {
    "rows": len(combined),
    "columns": combined.shape[1],
    "expected_rows_from_sources": int(row_count_table["row_count"].sum()),
    "invalid_datetime_rows": int(combined["Date"].isna().sum()),
    "duplicate_year_floor_date_rows": int(combined.duplicated(["year", "floor", "Date"]).sum()),
    "min_date": combined["Date"].min(),
    "max_date": combined["Date"].max(),
}

validation_summary_df = pd.DataFrame([validation_summary])
validation_summary_df


,rows,columns,expected_rows_from_sources,invalid_datetime_rows,duplicate_year_floor_date_rows,min_date,max_date
0,3162240,33,3162240,0,0,2018-07-01,2019-12-31 23:59:00


## 8. Save the merged output


In [8]:
OUTPUT_FILE = OUTPUT_DIR / "combined_wide_2018_2019_floors_3_4_5_7.csv.gz"
PREVIEW_FILE = OUTPUT_DIR / "combined_wide_2018_2019_floors_3_4_5_7_preview.csv"
SUMMARY_FILE = OUTPUT_DIR / "merge_summary_2018_2019_floors_3_4_5_7.csv"

combined.to_csv(OUTPUT_FILE, index=False, compression="gzip")
combined.head(1000).to_csv(PREVIEW_FILE, index=False)
validation_summary_df.to_csv(SUMMARY_FILE, index=False)

print("Merged output saved to:", OUTPUT_FILE)
print("Preview saved to:", PREVIEW_FILE)
print("Merge summary saved to:", SUMMARY_FILE)
print("Rows:", f"{len(combined):,}")
print("Columns:", combined.shape[1])


Merged output saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/combined_wide_2018_2019_floors_3_4_5_7.csv.gz
Preview saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/combined_wide_2018_2019_floors_3_4_5_7_preview.csv
Merge summary saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/merge_summary_2018_2019_floors_3_4_5_7.csv
Rows: 3,162,240
Columns: 33


## 9. Quick review


In [9]:
display(combined.head())
display(combined.groupby(["year", "floor"]).size().reset_index(name="rows"))
combined.shape

,year,floor,source_file,Date,z1_AC1(kW),z1_AC2(kW),z1_AC3(kW),z1_AC4(kW),z1_Light(kW),z1_Plug(kW),z1_S1(degC),z1_S1(RH%),z1_S1(lux),z2_AC1(kW),z2_Light(kW),z2_Plug(kW),z2_S1(degC),z2_S1(RH%),z2_S1(lux),z3_Light(kW),z3_Plug(kW),z4_AC1(kW),z4_Light(kW),z4_Plug(kW),z4_S1(degC),z4_S1(RH%),z4_S1(lux),z5_AC1(kW),z5_Light(kW),z5_Plug(kW),z5_S1(degC),z5_S1(RH%),z5_S1(lux)
0,2018,3,2018Floor3.csv,2018-07-01 00:00:00,0.0,0.0,0.0,0.07,0.0,0.03,28.44,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.01,0.26,0.0,0.0,0.02,27.69,65.66,0.0,0.0,0.0,0.02,26.88,71.16,0.0
1,2018,3,2018Floor3.csv,2018-07-01 00:01:00,0.0,0.0,0.0,0.07,0.0,0.03,28.44,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.02,0.37,0.0,0.0,0.02,27.69,65.65,0.0,0.0,0.0,0.02,26.88,71.16,0.0
2,2018,3,2018Floor3.csv,2018-07-01 00:02:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.01,0.32,0.0,0.0,0.02,27.69,65.63,0.0,0.0,0.0,0.02,26.88,71.18,0.0
3,2018,3,2018Floor3.csv,2018-07-01 00:03:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.66,0.0,0.0,0.0,0.03,27.67,63.36,0.0,0.01,0.26,0.0,0.0,0.02,NaN,NaN,NaN,0.0,0.0,0.02,26.88,71.19,0.0
4,2018,3,2018Floor3.csv,2018-07-01 00:04:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.66,0.0,0.0,0.0,0.03,27.67,63.36,0.0,0.02,0.37,0.0,0.0,0.02,NaN,NaN,NaN,0.0,0.0,0.02,26.88,71.21,0.0


,year,floor,rows
0,2018,3,264960
1,2018,4,264960
2,2018,5,264960
3,2018,7,264960
4,2019,3,525600
5,2019,4,525600
6,2019,5,525600
7,2019,7,525600


(3162240, 33)

## 10. Data cleaning workflow

The cleaning workflow is split into separate cells. Each cleaning step is wrapped in a function and returns an audit table so the notebook clearly shows what was changed.

The workflow uses the merged table named `combined` from the previous section.


In [10]:
# Cleaning configuration
RESAMPLE_INTERVAL = "15min"
EXPECTED_MINUTES_PER_WINDOW = 15

POWER_MIN_VALUE = 0
TEMPERATURE_MIN_C = 10
TEMPERATURE_MAX_C = 45
HUMIDITY_MIN_PCT = 0
HUMIDITY_MAX_PCT = 100
LUX_MIN_VALUE = 0

MEASUREMENT_COLUMNS = [col for col in combined.columns if col.startswith("z")]
POWER_COLUMNS = [col for col in MEASUREMENT_COLUMNS if col.endswith("(kW)")]
ZONE_IDS = sorted({int(match.group(1)) for col in MEASUREMENT_COLUMNS if (match := re.match(r"z(\d+)_", col))})
VALUE_COLUMNS = ["ac_power_kw", "light_power_kw", "plug_power_kw", "total_power_kw", "temperature_c", "humidity_pct", "lux"]
AVAILABILITY_COLUMNS = ["has_ac_meter", "has_light_meter", "has_plug_meter", "has_environment_sensor"]

print("Measurement columns:", len(MEASUREMENT_COLUMNS))
print("Power columns:", len(POWER_COLUMNS))
print("Zones:", ZONE_IDS)


Measurement columns: 29
Power columns: 17
Zones: [1, 2, 3, 4, 5]


## 11. Step 01: standardize timestamp

What this step does:

- Converts `Date` into a unified datetime column named `timestamp`.
- Counts invalid timestamps.
- Removes rows whose timestamp cannot be parsed.
- Returns how many rows were removed.


In [11]:
def clean_step_01_standardize_timestamp(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    before_rows = len(df)
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["Date"], errors="coerce")

    invalid_timestamp_rows = int(out["timestamp"].isna().sum())
    out = out.dropna(subset=["timestamp"]).reset_index(drop=True)

    audit = pd.DataFrame([{
        "step": "01_standardize_timestamp",
        "action": "Convert Date to timestamp and remove invalid timestamps",
        "rows_before": before_rows,
        "invalid_timestamp_rows": invalid_timestamp_rows,
        "rows_removed": before_rows - len(out),
        "rows_after": len(out),
        "min_timestamp": out["timestamp"].min(),
        "max_timestamp": out["timestamp"].max(),
    }])
    return out, audit

clean_step_01_df, audit_01_timestamp = clean_step_01_standardize_timestamp(combined)
display(audit_01_timestamp)


,step,action,rows_before,invalid_timestamp_rows,rows_removed,rows_after,min_timestamp,max_timestamp
0,01_standardize_timestamp,Convert Date to timestamp and remove invalid t...,3162240,0,0,3162240,2018-07-01,2019-12-31 23:59:00


## 12. Step 02: sort records

What this step does:

- Sorts the data by `floor`, `timestamp`, `year`, and `source_file`.
- Keeps the row count unchanged.
- Returns whether the output order matches the intended sorting keys.


In [12]:
def clean_step_02_sort_records(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    before_rows = len(df)
    sort_keys = ["floor", "timestamp", "year", "source_file"]
    out = df.sort_values(sort_keys).reset_index(drop=True)

    sorted_check = out[sort_keys].equals(out[sort_keys].sort_values(sort_keys).reset_index(drop=True))

    audit = pd.DataFrame([{
        "step": "02_sort_records",
        "action": "Sort by floor, timestamp, year, and source_file",
        "sort_keys": " -> ".join(sort_keys),
        "rows_before": before_rows,
        "rows_removed": 0,
        "rows_after": len(out),
        "is_sorted": bool(sorted_check),
    }])
    return out, audit

clean_step_02_df, audit_02_sort = clean_step_02_sort_records(clean_step_01_df)
display(audit_02_sort)


,step,action,sort_keys,rows_before,rows_removed,rows_after,is_sorted
0,02_sort_records,"Sort by floor, timestamp, year, and source_file",floor -> timestamp -> year -> source_file,3162240,0,3162240,True


## 13. Step 03: remove duplicate records

What this step does:

- Checks duplicate records using `year + floor + timestamp` as the key.
- Removes duplicate rows and keeps the first record.
- Returns how many duplicate rows were found and how many were removed.


In [13]:
def clean_step_03_remove_duplicates(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    before_rows = len(df)
    duplicate_keys = ["year", "floor", "timestamp"]
    duplicate_rows_found = int(df.duplicated(duplicate_keys).sum())

    out = df.drop_duplicates(duplicate_keys, keep="first").reset_index(drop=True)
    duplicate_rows_after = int(out.duplicated(duplicate_keys).sum())

    audit = pd.DataFrame([{
        "step": "03_remove_duplicates",
        "action": "Remove duplicate year-floor-timestamp records",
        "duplicate_keys": " + ".join(duplicate_keys),
        "rows_before": before_rows,
        "duplicate_rows_found": duplicate_rows_found,
        "duplicate_rows_removed": before_rows - len(out),
        "duplicate_rows_after": duplicate_rows_after,
        "rows_after": len(out),
    }])
    return out, audit

clean_step_03_df, audit_03_duplicates = clean_step_03_remove_duplicates(clean_step_02_df)
display(audit_03_duplicates)


,step,action,duplicate_keys,rows_before,duplicate_rows_found,duplicate_rows_removed,duplicate_rows_after,rows_after
0,03_remove_duplicates,Remove duplicate year-floor-timestamp records,year + floor + timestamp,3162240,0,0,0,3162240


## 14. Step 04: standardize numeric measurement columns

What this step does:

- Converts all sensor and meter columns to numeric values.
- Keeps units consistent with the original field names: power in kW, temperature in degC, humidity in RH%, and illuminance in lux.
- Counts values that became missing because they could not be parsed as numbers.


In [14]:
def clean_step_04_standardize_numeric_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    audit_rows = []

    for col in MEASUREMENT_COLUMNS:
        missing_before = int(out[col].isna().sum())
        converted = pd.to_numeric(out[col], errors="coerce")
        missing_after = int(converted.isna().sum())
        newly_coerced_to_missing = max(0, missing_after - missing_before)
        out[col] = converted

        audit_rows.append({
            "step": "04_standardize_numeric_columns",
            "action": "Convert measurement column to numeric",
            "column": col,
            "unit": "kW" if col.endswith("(kW)") else "degC" if col.endswith("(degC)") else "RH%" if col.endswith("(RH%)") else "lux" if col.endswith("(lux)") else "unknown",
            "missing_before": missing_before,
            "missing_after": missing_after,
            "newly_coerced_to_missing": newly_coerced_to_missing,
        })

    audit = pd.DataFrame(audit_rows)
    return out, audit

clean_step_04_df, audit_04_numeric = clean_step_04_standardize_numeric_columns(clean_step_03_df)
display(audit_04_numeric.head(10))
print("New values coerced to missing:", int(audit_04_numeric["newly_coerced_to_missing"].sum()))


,step,action,column,unit,missing_before,missing_after,newly_coerced_to_missing
0,04_standardize_numeric_columns,Convert measurement column to numeric,z1_AC1(kW),kW,258111,258111,0
1,04_standardize_numeric_columns,Convert measurement column to numeric,z1_AC2(kW),kW,258111,258111,0
2,04_standardize_numeric_columns,Convert measurement column to numeric,z1_AC3(kW),kW,258111,258111,0
3,04_standardize_numeric_columns,Convert measurement column to numeric,z1_AC4(kW),kW,211180,211180,0
4,04_standardize_numeric_columns,Convert measurement column to numeric,z1_Light(kW),kW,153882,153882,0
5,04_standardize_numeric_columns,Convert measurement column to numeric,z1_Plug(kW),kW,102211,102211,0
6,04_standardize_numeric_columns,Convert measurement column to numeric,z1_S1(degC),degC,1151289,1151289,0
7,04_standardize_numeric_columns,Convert measurement column to numeric,z1_S1(RH%),RH%,1151286,1151286,0
8,04_standardize_numeric_columns,Convert measurement column to numeric,z1_S1(lux),lux,1151347,1151347,0
9,04_standardize_numeric_columns,Convert measurement column to numeric,z2_AC1(kW),kW,211098,211098,0


New values coerced to missing: 0


## 15. Step 05: check impossible values

What this step does:

- Checks negative power values.
- Checks temperature outside `10-45` degC.
- Checks humidity outside `0-100` RH%.
- Checks negative lux values.
- Converts impossible values to missing values.
- Returns how many impossible values were found by column and rule.


In [15]:
def clean_step_05_check_impossible_values(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    audit_rows = []

    rules = []
    for col in MEASUREMENT_COLUMNS:
        if col.endswith("(kW)"):
            rules.append((col, "power_kw < 0", out[col] < POWER_MIN_VALUE))
        elif col.endswith("(degC)"):
            rules.append((col, "temperature_c outside 10-45", (out[col] < TEMPERATURE_MIN_C) | (out[col] > TEMPERATURE_MAX_C)))
        elif col.endswith("(RH%)"):
            rules.append((col, "humidity_pct outside 0-100", (out[col] < HUMIDITY_MIN_PCT) | (out[col] > HUMIDITY_MAX_PCT)))
        elif col.endswith("(lux)"):
            rules.append((col, "lux < 0", out[col] < LUX_MIN_VALUE))

    for col, rule_name, invalid_mask in rules:
        invalid_count = int(invalid_mask.sum())
        if invalid_count:
            out.loc[invalid_mask, col] = pd.NA
        audit_rows.append({
            "step": "05_check_impossible_values",
            "action": "Convert impossible values to missing",
            "column": col,
            "invalid_rule": rule_name,
            "invalid_count": invalid_count,
            "values_changed_to_missing": invalid_count,
        })

    audit = pd.DataFrame(audit_rows)
    return out, audit

clean_step_05_df, audit_05_invalid_values = clean_step_05_check_impossible_values(clean_step_04_df)
display(audit_05_invalid_values.query("invalid_count > 0"))
print("Total impossible values found:", int(audit_05_invalid_values["invalid_count"].sum()))


,step,action,column,invalid_rule,invalid_count,values_changed_to_missing


Total impossible values found: 0


## 16. Step 06: convert wide table to zone-level table

What this step does:

- Converts the wide floor-level table into one row per `timestamp + floor + zone`.
- Aggregates multiple AC meters within a zone into `ac_power_kw`.
- Keeps lighting, plug, temperature, humidity, and lux as standardized fields.
- Adds meter/sensor availability flags.
- Preserves real `0` values and keeps missing values as `NaN`.


In [16]:
def clean_step_06_convert_to_zone_level(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    zone_frames = []
    audit_rows = []

    for (year, floor, source_file), group in df.groupby(["year", "floor", "source_file"], sort=True):
        group = group.sort_values("timestamp").reset_index(drop=True)

        for zone in ZONE_IDS:
            prefix = f"z{zone}_"
            ac_cols = [col for col in group.columns if re.fullmatch(rf"{prefix}AC\d+\(kW\)", col)]
            light_col = f"{prefix}Light(kW)"
            plug_col = f"{prefix}Plug(kW)"
            temp_col = f"{prefix}S1(degC)"
            humidity_col = f"{prefix}S1(RH%)"
            lux_col = f"{prefix}S1(lux)"
            missing_numeric = pd.Series(pd.NA, index=group.index, dtype="Float64")

            zone_df = pd.DataFrame({
                "timestamp": group["timestamp"],
                "year": year,
                "floor": floor,
                "zone": zone,
                "source_file": source_file,
                "has_ac_meter": bool(ac_cols),
                "has_light_meter": light_col in group.columns,
                "has_plug_meter": plug_col in group.columns,
                "has_environment_sensor": all(col in group.columns for col in [temp_col, humidity_col, lux_col]),
            })
            zone_df["ac_power_kw"] = group[ac_cols].sum(axis=1, min_count=1) if ac_cols else missing_numeric.copy()
            zone_df["light_power_kw"] = group[light_col] if light_col in group.columns else missing_numeric.copy()
            zone_df["plug_power_kw"] = group[plug_col] if plug_col in group.columns else missing_numeric.copy()
            zone_df["temperature_c"] = group[temp_col] if temp_col in group.columns else missing_numeric.copy()
            zone_df["humidity_pct"] = group[humidity_col] if humidity_col in group.columns else missing_numeric.copy()
            zone_df["lux"] = group[lux_col] if lux_col in group.columns else missing_numeric.copy()
            zone_df["total_power_kw"] = zone_df[["ac_power_kw", "light_power_kw", "plug_power_kw"]].sum(axis=1, min_count=1)

            zone_frames.append(zone_df)
            audit_rows.append({
                "step": "06_convert_to_zone_level",
                "action": "Convert wide floor-level data to timestamp-floor-zone rows",
                "year": year,
                "floor": floor,
                "zone": zone,
                "source_file": source_file,
                "source_rows": len(group),
                "zone_rows_created": len(zone_df),
                "ac_meter_count": len(ac_cols),
                "has_light_meter": light_col in group.columns,
                "has_plug_meter": plug_col in group.columns,
                "has_environment_sensor": all(col in group.columns for col in [temp_col, humidity_col, lux_col]),
            })

    out = pd.concat(zone_frames, ignore_index=True)
    audit = pd.DataFrame(audit_rows)
    return out, audit

zone_level_minute_df, audit_06_zone_level = clean_step_06_convert_to_zone_level(clean_step_05_df)
display(audit_06_zone_level.head(10))
print("Zone-level minute rows created:", f"{len(zone_level_minute_df):,}")


/var/folders/p4/f4s_7smj75v4r3f2mdqbl2j40000gn/T/ipykernel_13618/3886857628.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(zone_frames, ignore_index=True)


,step,action,year,floor,zone,source_file,source_rows,zone_rows_created,ac_meter_count,has_light_meter,has_plug_meter,has_environment_sensor
0,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,1,2018Floor3.csv,264960,264960,4,True,True,True
1,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,2,2018Floor3.csv,264960,264960,1,True,True,True
2,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,3,2018Floor3.csv,264960,264960,0,True,True,False
3,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,4,2018Floor3.csv,264960,264960,1,True,True,True
4,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,3,5,2018Floor3.csv,264960,264960,1,True,True,True
5,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,1,2018Floor4.csv,264960,264960,4,True,True,True
6,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,2,2018Floor4.csv,264960,264960,1,True,True,True
7,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,3,2018Floor4.csv,264960,264960,0,True,True,False
8,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,4,2018Floor4.csv,264960,264960,1,True,True,True
9,06_convert_to_zone_level,Convert wide floor-level data to timestamp-flo...,2018,4,5,2018Floor4.csv,264960,264960,1,True,True,True


Zone-level minute rows created: 15,811,200


## 17. Step 07: sort zone-level records

What this step does:

- Sorts the modelling table by `floor -> zone -> timestamp`.
- Keeps row count unchanged.
- Returns whether the final order matches the expected modelling order.


In [17]:
def clean_step_07_sort_zone_records(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    before_rows = len(df)
    sort_keys = ["floor", "zone", "timestamp"]
    out = df.sort_values(sort_keys).reset_index(drop=True)
    sorted_check = out[sort_keys].equals(out[sort_keys].sort_values(sort_keys).reset_index(drop=True))

    audit = pd.DataFrame([{
        "step": "07_sort_zone_records",
        "action": "Sort zone-level records by floor, zone, and timestamp",
        "sort_keys": " -> ".join(sort_keys),
        "rows_before": before_rows,
        "rows_removed": 0,
        "rows_after": len(out),
        "is_sorted": bool(sorted_check),
    }])
    return out, audit

zone_level_minute_sorted_df, audit_07_zone_sort = clean_step_07_sort_zone_records(zone_level_minute_df)
display(audit_07_zone_sort)


,step,action,sort_keys,rows_before,rows_removed,rows_after,is_sorted
0,07_sort_zone_records,"Sort zone-level records by floor, zone, and ti...",floor -> zone -> timestamp,15811200,0,15811200,True


## 18. Step 08: resample to 15-minute intervals

What this step does:

- Resamples each `year + floor + zone` group to 15-minute intervals.
- Calculates mean, min, max, and valid-count features for each measurement.
- Counts missing time windows and incomplete 15-minute windows.
- Does not fill missing measurement values with `0`.


In [18]:
def clean_step_08_resample_15min(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    resampled_parts = []
    audit_rows = []

    for (year, floor, zone), group in df.groupby(["year", "floor", "zone"], sort=True):
        group = group.sort_values("timestamp").set_index("timestamp")
        start = group.index.min().floor(RESAMPLE_INTERVAL)
        end = group.index.max().floor(RESAMPLE_INTERVAL)
        full_index = pd.date_range(start=start, end=end, freq=RESAMPLE_INTERVAL)

        value_mean = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).mean().reindex(full_index)
        value_min = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).min().reindex(full_index).add_suffix("_min")
        value_max = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).max().reindex(full_index).add_suffix("_max")
        valid_counts = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).count().reindex(full_index, fill_value=0).add_suffix("_valid_count")
        observed_minutes = group["source_file"].resample(RESAMPLE_INTERVAL).count().reindex(full_index, fill_value=0)

        out = pd.concat([value_mean, value_min, value_max, valid_counts], axis=1)
        out.insert(0, "timestamp", full_index)
        out.insert(1, "year", year)
        out.insert(2, "floor", floor)
        out.insert(3, "zone", zone)
        out["observed_minutes"] = observed_minutes.to_numpy()
        out["missing_time_flag"] = out["observed_minutes"] == 0
        out["incomplete_window_flag"] = out["observed_minutes"] < EXPECTED_MINUTES_PER_WINDOW

        for col in AVAILABILITY_COLUMNS:
            out[col] = bool(group[col].iloc[0])

        resampled_parts.append(out.reset_index(drop=True))
        audit_rows.append({
            "step": "08_resample_15min",
            "action": "Resample each year-floor-zone group to 15-minute intervals",
            "year": year,
            "floor": floor,
            "zone": zone,
            "minute_rows_before": len(group),
            "rows_after_resample": len(out),
            "missing_time_windows": int(out["missing_time_flag"].sum()),
            "incomplete_15min_windows": int(out["incomplete_window_flag"].sum()),
            "min_timestamp": out["timestamp"].min(),
            "max_timestamp": out["timestamp"].max(),
        })

    cleaned = pd.concat(resampled_parts, ignore_index=True)
    cleaned = cleaned.sort_values(["floor", "zone", "timestamp"]).reset_index(drop=True)
    audit = pd.DataFrame(audit_rows)
    return cleaned, audit

cleaned_15min_without_measurement_flags, audit_08_resample = clean_step_08_resample_15min(zone_level_minute_sorted_df)
display(audit_08_resample.head(10))
print("15-minute rows:", f"{len(cleaned_15min_without_measurement_flags):,}")
print("Missing time windows:", int(audit_08_resample["missing_time_windows"].sum()))
print("Incomplete 15-minute windows:", int(audit_08_resample["incomplete_15min_windows"].sum()))


,step,action,year,floor,zone,minute_rows_before,rows_after_resample,missing_time_windows,incomplete_15min_windows,min_timestamp,max_timestamp
0,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,1,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
1,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,2,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
2,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,3,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
3,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,4,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
4,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,3,5,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
5,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,1,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
6,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,2,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
7,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,3,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
8,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,4,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00
9,08_resample_15min,Resample each year-floor-zone group to 15-minu...,2018,4,5,264960,17664,0,0,2018-07-01,2018-12-31 23:45:00


15-minute rows: 1,054,080
Missing time windows: 0
Incomplete 15-minute windows: 0


## 19. Step 09: add measurement missing flags

What this step does:

- Adds one missing flag for each main measurement column.
- Keeps real `0` values unchanged.
- Returns missing counts and percentages for each measurement.


In [19]:
def clean_step_09_add_measurement_missing_flags(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    audit_rows = []

    for col in VALUE_COLUMNS:
        flag_col = f"{col}_missing_flag"
        out[flag_col] = out[col].isna()
        audit_rows.append({
            "step": "09_add_measurement_missing_flags",
            "action": "Add missing flag for measurement column",
            "measurement_column": col,
            "flag_column": flag_col,
            "missing_count": int(out[flag_col].sum()),
            "missing_pct": float(out[flag_col].mean() * 100),
            "zero_count_preserved": int((out[col] == 0).sum(skipna=True)),
        })

    audit = pd.DataFrame(audit_rows)
    return out, audit

cleaned_15min, audit_09_missing_flags = clean_step_09_add_measurement_missing_flags(cleaned_15min_without_measurement_flags)
display(audit_09_missing_flags)


,step,action,measurement_column,flag_column,missing_count,missing_pct,zero_count_preserved
0,09_add_measurement_missing_flags,Add missing flag for measurement column,ac_power_kw,ac_power_kw_missing_flag,259233,24.593295,564649
1,09_add_measurement_missing_flags,Add missing flag for measurement column,light_power_kw,light_power_kw_missing_flag,41946,3.979394,363933
2,09_add_measurement_missing_flags,Add missing flag for measurement column,plug_power_kw,plug_power_kw_missing_flag,38413,3.644221,36102
3,09_add_measurement_missing_flags,Add missing flag for measurement column,total_power_kw,total_power_kw_missing_flag,33365,3.165320,23394
4,09_add_measurement_missing_flags,Add missing flag for measurement column,temperature_c,temperature_c_missing_flag,515539,48.908906,0
5,09_add_measurement_missing_flags,Add missing flag for measurement column,humidity_pct,humidity_pct_missing_flag,515539,48.908906,0
6,09_add_measurement_missing_flags,Add missing flag for measurement column,lux,lux_missing_flag,515543,48.909286,256692


## 20. Step 10: final validation

What this step does:

- Checks final row and column counts.
- Confirms there are no duplicate `floor + zone + timestamp` records.
- Confirms selected floors and zones.
- Summarizes all cleaning audit outputs.


In [20]:
def clean_step_10_final_validation(cleaned_df: pd.DataFrame, audit_tables: list[pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame]:
    duplicate_final_rows = int(cleaned_df.duplicated(["floor", "zone", "timestamp"]).sum())
    final_audit = pd.DataFrame([{
        "step": "10_final_validation",
        "action": "Validate cleaned 15-minute zone-level table",
        "rows": len(cleaned_df),
        "columns": cleaned_df.shape[1],
        "floors": ",".join(map(str, sorted(cleaned_df["floor"].unique()))),
        "zones": ",".join(map(str, sorted(cleaned_df["zone"].unique()))),
        "min_timestamp": cleaned_df["timestamp"].min(),
        "max_timestamp": cleaned_df["timestamp"].max(),
        "duplicate_floor_zone_timestamp_rows": duplicate_final_rows,
        "missing_time_windows": int(cleaned_df["missing_time_flag"].sum()),
        "incomplete_15min_windows": int(cleaned_df["incomplete_window_flag"].sum()),
    }])

    all_audits = pd.concat(audit_tables + [final_audit], ignore_index=True, sort=False)
    return final_audit, all_audits

final_validation_summary, all_cleaning_audits = clean_step_10_final_validation(
    cleaned_15min,
    [
        audit_01_timestamp,
        audit_02_sort,
        audit_03_duplicates,
        audit_04_numeric,
        audit_05_invalid_values,
        audit_06_zone_level,
        audit_07_zone_sort,
        audit_08_resample,
        audit_09_missing_flags,
    ],
)

display(final_validation_summary)


,step,action,rows,columns,floors,zones,min_timestamp,max_timestamp,duplicate_floor_zone_timestamp_rows,missing_time_windows,incomplete_15min_windows
0,10_final_validation,Validate cleaned 15-minute zone-level table,1054080,46,"3,4,5,7","1,2,3,4,5",2018-07-01,2019-12-31 23:45:00,0,0,0


## 21. Step 11: save cleaned outputs

What this step does:

- Saves the complete cleaned 15-minute table locally as `.csv.gz`.
- Saves a small preview CSV for GitHub review.
- Saves audit, invalid-value, duplicate, and missing-value summaries.


In [21]:
def clean_step_11_save_cleaning_outputs(cleaned_df: pd.DataFrame) -> pd.DataFrame:
    output_paths = {
        "cleaned_15min_file": OUTPUT_DIR / "cleaned_zone_15min_2018_2019_floors_3_4_5_7.csv.gz",
        "cleaned_preview_file": OUTPUT_DIR / "cleaned_zone_15min_2018_2019_floors_3_4_5_7_preview.csv",
        "all_cleaning_audits_file": OUTPUT_DIR / "all_cleaning_audits_2018_2019_floors_3_4_5_7.csv",
        "cleaning_summary_file": OUTPUT_DIR / "cleaning_summary_2018_2019_floors_3_4_5_7.csv",
        "duplicate_summary_file": OUTPUT_DIR / "duplicate_summary_2018_2019_floors_3_4_5_7.csv",
        "invalid_values_summary_file": OUTPUT_DIR / "invalid_values_summary_2018_2019_floors_3_4_5_7.csv",
        "missing_summary_file": OUTPUT_DIR / "missing_summary_2018_2019_floors_3_4_5_7.csv",
        "final_validation_summary_file": OUTPUT_DIR / "final_validation_summary_2018_2019_floors_3_4_5_7.csv",
    }

    cleaned_df.to_csv(output_paths["cleaned_15min_file"], index=False, compression="gzip")
    cleaned_df.head(1000).to_csv(output_paths["cleaned_preview_file"], index=False)
    all_cleaning_audits.to_csv(output_paths["all_cleaning_audits_file"], index=False)
    all_cleaning_audits.to_csv(output_paths["cleaning_summary_file"], index=False)
    audit_03_duplicates.to_csv(output_paths["duplicate_summary_file"], index=False)
    audit_05_invalid_values.to_csv(output_paths["invalid_values_summary_file"], index=False)
    audit_09_missing_flags.to_csv(output_paths["missing_summary_file"], index=False)
    final_validation_summary.to_csv(output_paths["final_validation_summary_file"], index=False)

    save_audit = pd.DataFrame([{
        "step": "11_save_cleaning_outputs",
        "action": "Save cleaned table, preview, and audit outputs",
        "output_name": name,
        "output_path": str(path),
        "file_exists": path.exists(),
        "file_size_mb": path.stat().st_size / 1024 / 1024 if path.exists() else 0,
    } for name, path in output_paths.items()])
    return save_audit

save_audit = clean_step_11_save_cleaning_outputs(cleaned_15min)
display(save_audit)


,step,action,output_name,output_path,file_exists,file_size_mb
0,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaned_15min_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,27.858608
1,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaned_preview_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.283359
2,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",all_cleaning_audits_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.024652
3,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",cleaning_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.024652
4,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",duplicate_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.000218
5,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",invalid_values_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.002723
6,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",missing_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.001028
7,11_save_cleaning_outputs,"Save cleaned table, preview, and audit outputs",final_validation_summary_file,/Users/yonglijiang/Documents/Adelaide Universi...,True,0.000269
